In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .enableHiveSupport()\
    .appName("test") \
    .getOrCreate()

In [14]:
spark

In [15]:
from pyspark.sql import functions as F

In [16]:
users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)
orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)
products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)

In [17]:
users.show()

+-------+-------+
|user_id|   city|
+-------+-------+
|     u1| Berlin|
|     u2| Berlin|
|     u3| Munich|
|     u4|Hamburg|
+-------+-------+



In [22]:
orders.show()

+--------+-------+----------+---+-----+
|order_id|user_id|product_id|qty|price|
+--------+-------+----------+---+-----+
|      o1|     u1|        p1|  2| 10.0|
|      o2|     u1|        p2|  1| 30.0|
|      o3|     u2|        p1|  1| 10.0|
|      o4|     u2|        p3|  5|  7.0|
|      o5|     u3|        p2|  3| 30.0|
|      o6|     u3|        p3|  1|  7.0|
|      o7|     u4|        p1| 10| 10.0|
+--------+-------+----------+---+-----+



In [23]:
products.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|        p1|   Ring VOLA|
|        p2|  Ring POROG|
|        p3|Ring TISHINA|
+----------+------------+



## Соберем витрину mart_city_top_products:

### 1. Посчитать revenue = qty * price

In [25]:
orders = orders.withColumn("revenue", F.col("qty")*F.col("price"))

In [26]:
orders.show()

+--------+-------+----------+---+-----+-------+
|order_id|user_id|product_id|qty|price|revenue|
+--------+-------+----------+---+-----+-------+
|      o1|     u1|        p1|  2| 10.0|   20.0|
|      o2|     u1|        p2|  1| 30.0|   30.0|
|      o3|     u2|        p1|  1| 10.0|   10.0|
|      o4|     u2|        p3|  5|  7.0|   35.0|
|      o5|     u3|        p2|  3| 30.0|   90.0|
|      o6|     u3|        p3|  1|  7.0|    7.0|
|      o7|     u4|        p1| 10| 10.0|  100.0|
+--------+-------+----------+---+-----+-------+



### Объединить orders с users и products

In [52]:
ord_users = orders.join(users, on = "user_id", how = "left")
ord_users_prod = ord_users.join(products, on = "product_id", how = "left")

In [53]:
ord_users_prod.show()

+----------+-------+--------+---+-----+-------+-------+------------+
|product_id|user_id|order_id|qty|price|revenue|   city|product_name|
+----------+-------+--------+---+-----+-------+-------+------------+
|        p2|     u1|      o2|  1| 30.0|   30.0| Berlin|  Ring POROG|
|        p2|     u3|      o5|  3| 30.0|   90.0| Munich|  Ring POROG|
|        p1|     u1|      o1|  2| 10.0|   20.0| Berlin|   Ring VOLA|
|        p1|     u2|      o3|  1| 10.0|   10.0| Berlin|   Ring VOLA|
|        p1|     u4|      o7| 10| 10.0|  100.0|Hamburg|   Ring VOLA|
|        p3|     u2|      o4|  5|  7.0|   35.0| Berlin|Ring TISHINA|
|        p3|     u3|      o6|  1|  7.0|    7.0| Munich|Ring TISHINA|
+----------+-------+--------+---+-----+-------+-------+------------+



### Посчитать метрики по (city, product_id, product_name):
- orders_cnt (кол-во заказов)
- qty_sum (сумма qty)
- revenue_sum (сумма revenue)

In [95]:
metrics = (
    ord_users_prod
    .groupBy("city", "product_id", "product_name")
    .agg(
        F.count("*").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
                           
    )
)
metrics.show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p2|  Ring POROG|         1|      1|       30.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+



### Для каждого города выбрать Top-2 товара по revenue_sum используя Window


In [69]:
from pyspark.sql.window import Window

In [101]:
w = Window.partitionBy("city").orderBy(F.col("revenue_sum").desc())

top_2 = (
    metrics
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 2)
    .drop("rn")
)

In [102]:
top_2.show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p2|  Ring POROG|         1|      1|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+



### 5. Записать результат

In [103]:
top_2.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("result")

### 6. Прочитать обратно и показать топ-результат.

In [106]:
df = spark.read.csv("result/result.csv", header = True, inferSchema=True)

In [107]:
df.show(1)

+------+----------+------------+----------+-------+-----------+
|  city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+------+----------+------------+----------+-------+-----------+
|Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
+------+----------+------------+----------+-------+-----------+
only showing top 1 row

